In [20]:
#from cellpose import models, io
#import pyclesperanto as cle
from bioio import BioImage
import napari
from skimage.transform import rescale, resize
from skimage.io import imsave, imread
from skimage import exposure, morphology, segmentation, util
from skimage.measure import regionprops, regionprops_table, marching_cubes, mesh_surface_area
import numpy as np
import random
from glob import glob
import os

#cle.select_device("NVIDIA")

In [ ]:
def normalize_images(input_image,tophat_radius):
    input_gpu = cle.push(input_image)
    #normalizing the image stack
    equalized_intensities_stack = cle.create_like(input_gpu)
    a_slice = cle.create([input_gpu.shape[1], input_gpu.shape[2]])
    num_slices = input_gpu.shape[0]
    mean_intensity_stack = cle.mean_of_all_pixels(input_gpu)
    corrected_slice = None
    for z in range(0, num_slices):
        # get a single slice out of the stack
        cle.copy_slice(input_gpu, a_slice, z)
        # measure its intensity
        mean_intensity_slice = cle.mean_of_all_pixels(a_slice)
        # correct the intensity
        correction_factor = mean_intensity_slice/mean_intensity_stack
        corrected_slice = cle.multiply_image_and_scalar(a_slice, corrected_slice, correction_factor)
        # copy slice back in a stack
        cle.copy_slice(corrected_slice, equalized_intensities_stack, z)
    #background subtraction (increase the signal to noise ratio for improved segmentation results)
    background_subtracted_top_hat = cle.top_hat_sphere(equalized_intensities_stack,radius_x=tophat_radius,radius_y=tophat_radius,radius_z=tophat_radius)
    #pull data off gpu
    input_pull = cle.pull(input_gpu)
    background_subtracted_top_hat_pull = cle.pull(background_subtracted_top_hat)
    equalized_intensities_stack_pull = cle.pull(equalized_intensities_stack)
    return background_subtracted_top_hat_pull

def get_3D_surfacearea_and_sphericity(mask_img,df,voxel):
    surf_area_list = []
    sphericity_list = []
    vol = np.asarray(df['area']).astype(np.float64)
    c = 0
    for i in np.asarray(df['label']).astype(int):
        obj = mask_img == i
        verts, faces, _ , _ = sk.measure.marching_cubes(obj, level=0.0,spacing=voxel)
        surf_area = sk.measure.mesh_surface_area(verts, faces)
        surf_area_list.append(surf_area)
        sphericity = (np.pi**(1/3)*((6*vol[c])**(2/3)))/surf_area
        sphericity_list.append(sphericity)
        c += 1
    surface_areas = pd.Series(surf_area_list,name='Surface_Area (um^2)')
    sphericities = pd.Series(sphericity_list,name='Sphericity')
    merged_df = pd.concat([df,surface_areas,sphericities], axis=1)
    return merged_df

# Testing parameters for cellpose 4

In [ ]:
test_img = r"E:\KristinG\20260625_zebrafish\fin\WT_ctrl_2_skin_3_dorsal_fin_Out.czi"
img = BioImage(test_img)
img_rd = img.get_image_data("ZYX", C=1)

In [ ]:
img_twoch = img.get_image_data("ZYXC")

In [ ]:
img_twoch.shape

In [ ]:
cp = io.logger_setup()
model = models.CellposeModel(gpu=True)

In [ ]:
rescaled_img = rescale(img_rd, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True)

In [ ]:
new_scale = img.physical_pixel_sizes[0]*0.5, img.physical_pixel_sizes[1]*0.2, img.physical_pixel_sizes[2]*0.2

In [ ]:
anisotropy = new_scale[0]/new_scale[1]

In [ ]:
rescaled_masks_stitch, rescaled_flows_stitch, _ = model.eval(rescaled_img, z_axis=0, batch_size=32, do_3D=False,normalize=True,stitch_threshold=0.7,anisotropy=anisotropy)

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(rescaled_img, name='rescaled image', scale=new_scale,channel_axis=None)
v.add_labels(rescaled_masks_stitch, name='rescaled masks stitch 0.7 norm anisotropy', opacity=0.5,scale=new_scale)

In [ ]:
props = regionprops_table(rescaled_masks_stitch, properties=['label'])

In [ ]:
#v2.add_image(rescaled_img, name='rescaled image', scale=(img.physical_pixel_sizes[0]*(200/30), img.physical_pixel_sizes[1]*(200/30), img.physical_pixel_sizes[2]*(200/30)))
random_subset = []
while len(random_subset) < 10:
    r = randint(1, len(props['label']))
    if r not in random_subset:
        random_subset.append(r)

for i in random_subset:
    l = props['label'][i]
    obj = rescaled_masks_stitch == l
    v.add_labels(obj, name=f'object {l}', scale=new_scale, opacity=0.5)

In [ ]:
for l in props['equivalent_diameter_area']:
    print(l)

In [ ]:
save = r"E:\KristinG\20260625_zebrafish\fin_predictions"
imsave(r"E:\KristinG\20260625_zebrafish\fin_rescaled\DKO_ctrl_1_skin_1_ventral_fin_Out_rescaled.tif", rescaled_img.astype('uint16'))
imsave(r"E:\KristinG\20260625_zebrafish\fin_predictions\DKO_ctrl_1_skin_1_ventral_fin_Out_rescaled_masks_stitch.tif", rescaled_masks_stitch.astype('uint16'))

# Run cellpose4 on random subset of imgs

In [ ]:
img_files = sorted(glob(r"E:\KristinG\20260625_zebrafish\fin\*.czi"))
wt_files = [f for f in img_files if "WT" in f]
dko_files = [f for f in img_files if "DKO" in f]
rand_WT = random.sample(wt_files, 5)
rand_DKO = random.sample(dko_files, 5)
all_files = rand_WT + rand_DKO

In [ ]:
imgs = [BioImage(f).get_image_data("ZYX", C=1) for f in all_files]

In [ ]:
pixel_spacing = (BioImage(all_files[0]).physical_pixel_sizes[0],BioImage(all_files[0]).physical_pixel_sizes[1],BioImage(all_files[0]).physical_pixel_sizes[2])

In [ ]:
test_file = all_files[0]
test_data = BioImage(test_file)

In [ ]:
test_data.dims['Z'][0]

In [ ]:
pixel_spacing

In [ ]:
norm_imgs = [normalize_images(img, tophat_radius=10) for img in imgs]

In [ ]:
equal_adpt_imgs = [exposure.equalize_adapthist(img) for img in imgs]


In [ ]:
equalized_hist = [exposure.equalize_hist(img) for img in imgs] #not a good fit for this data

In [ ]:
del(equalized_hist)

In [ ]:
#v = napari.Viewer()
v.add_image(imgs[3], name='original image', scale=pixel_spacing,channel_axis=None)
v.add_image(norm_imgs[3], name='normalized image', scale=pixel_spacing,channel_axis=None)
v.add_image(equal_adpt_imgs[3], name='equalized adaptive image', scale=pixel_spacing,channel_axis=None)
v.add_image(equalized_hist[3], name='equalized histogram image', scale=pixel_spacing,channel_axis=None)

In [ ]:
rescaled_og_imgs = [rescale(img, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True) for img in imgs]
rescaled_norm_imgs = [rescale(img, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True) for img in norm_imgs]
rescaled_adpt_imgs = [rescale(img, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True) for img in equal_adpt_imgs]

In [ ]:
new_scale = (pixel_spacing[0]*0.5, pixel_spacing[1]*0.2, pixel_spacing[2]*0.2)

In [ ]:
#adjusting parameters for minimizing artifacts
rand_test_set = sorted(glob(r"E:\KristinG\20260625_zebrafish\fin_rescaled\*.tif"))
rand_test_imgs = list(map(imread, rand_test_set))

In [ ]:
io.logger_setup()
model = models.CellposeModel(gpu=True)

In [ ]:
#og_masks, _, _ = model.eval(rescaled_og_imgs, batch_size=5, z_axis=0, do_3D=False,normalize=True,stitch_threshold=0.65)
norm_masks, _, _ = model.eval(rescaled_norm_imgs, batch_size=5, z_axis=0, do_3D=False,normalize=True,stitch_threshold=0.6,cellprob_threshold=-0.5)
#adpt_masks, _, _ = model.eval(rescaled_adpt_imgs, batch_size=5, z_axis=0, do_3D=False,normalize=True,stitch_threshold=0.65)

In [ ]:
del(og_masks,adpt_masks)

In [ ]:
v.add_image(rescaled_og_imgs[1], name='original image', scale=new_scale,channel_axis=None)
#v.add_labels(og_masks[3], name='original image masks', scale=new_scale)
v.add_labels(norm_masks[1], name='normalized image masks', scale=new_scale)
#v.add_labels(adpt_masks[3], name='equalized adaptive image masks', scale=new_scale)

In [ ]:
# resize masks to match original images
resized_masks = []
for m, i in zip(norm_masks,imgs):
    new_size = resize(m,i.shape,preserve_range=True,order=0).astype(np.uint16)
    resized_masks.append(new_size)

In [ ]:
resized_masks[0].dtype

In [ ]:
v.add_image(imgs[1], name='original image', scale=pixel_spacing,channel_axis=None)
v.add_labels(resized_masks[1], name='resized image masks', scale=pixel_spacing)

In [ ]:
for i, img in enumerate(zip(masks, all_imgs)):
    m, img = img
    save = r"E:\KristinG\20260625_zebrafish\fin_predictions"
    imsave(rf"E:\KristinG\20260625_zebrafish\fin_predictions\{i}_rescaled_masks_stitch.tif", m.astype('uint16'))
    imsave(rf"E:\KristinG\20260625_zebrafish\fin_rescaled\{i}_rescaled.tif", img.astype('uint16'))

## Getting Measurements

In [ ]:
import skimage as sk

In [ ]:
clear_boarder_test = sk.segmentation.clear_border(resized_masks[0])

In [ ]:
v = napari.Viewer()
v.add_labels(clear_boarder_test,name='cleared',scale=pixel_spacing)
v.add_labels(resized_masks[0],name='og',scale=pixel_spacing)


In [ ]:
shape = resized_masks[0].shape

In [ ]:
shape

In [ ]:
# clear only the XY boarders of objects that touch
bool_arr = np.zeros((shape[0],shape[1]-12,shape[2]-12))

In [ ]:
xy_boarder = np.pad(bool_arr,pad_width=((0,0),(6,6),(6,6)),mode='constant',constant_values=1).astype(np.bool)

In [ ]:
#v = napari.Viewer()
v.add_image(xy_boarder,scale=pixel_spacing)

In [ ]:
test_mask = resized_masks[0]

In [ ]:
labels, number = sk.measure.label(test_mask,background=0,return_num=True)
indices = np.arange(number+1)

In [ ]:
border_indices = np.unique(test_mask[xy_boarder])

In [ ]:
label_mask = np.isin(indices, border_indices)

In [ ]:
labels.shape

In [ ]:
label_mask

In [ ]:
mask = label_mask[labels.reshape(-1)].reshape(labels.shape)

In [ ]:
masked_labels = test_mask

In [ ]:
masked_labels[mask] = 0

In [ ]:
v = napari.Viewer()
v.add_image(xy_boarder,scale=pixel_spacing)
v.add_labels(labels,name='original labels',scale=pixel_spacing)
v.add_labels(masked_labels,name='cleared xy borders',scale=pixel_spacing)

# Validate cell predictions and screen for needed adjustments

In [2]:
path = r"E:\KristinG\20260625_zebrafish\fin"
img_files = sorted(glob(rf'{path}\*.czi'))
cell_pred_path = r"E:\KristinG\20260625_zebrafish\fin_outputs\Cell_Masks"
cell_pred_files = sorted(glob(rf'{cell_pred_path}\*.tif'))
img = BioImage(img_files[0])
pixel_spacing = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])

In [3]:
v = napari.Viewer()

In [4]:
img_dict = {}
for i, p in enumerate(zip(img_files,cell_pred_files)):
    img, mask = p
    img_dict[i] = {'img': img, 'mask': mask}

In [30]:
rand_samp = random.sample(range(0,len(img_files)), 3)

In [31]:
imgs = []
masks = []
for i in rand_samp:
    imgs.append(BioImage(img_dict[i]['img']).get_image_data("ZYX", C=1))
    masks.append(imread(img_dict[i]['mask']))
    #v.add_image(rand_img,name=f'Img {i}',scale=pixel_spacing)
    #v.add_labels(rand_cells,name=f'Mask {i}',scale=pixel_spacing)


In [33]:
removed_small_obj = [morphology.remove_small_objects(m,max_size=200) for m in masks]


In [34]:
boundaries = [(segmentation.find_boundaries(m,connectivity=m.ndim,mode='outer').astype(np.int8) - 1)*-1 for m in removed_small_obj]


In [35]:
removed_small_holes = [morphology.remove_small_holes(m,max_size=200) for m in removed_small_obj]


c:\Users\gkristin.ATKK\AppData\Local\miniconda3\envs\cellpose4\Lib\site-packages\skimage\_shared\utils.py:386: UserWarning: Any labeled images will be returned as a boolean array. Did you mean to use a boolean array?
  return func(*args, **kwargs)
c:\Users\gkristin.ATKK\AppData\Local\miniconda3\envs\cellpose4\Lib\site-packages\skimage\_shared\utils.py:386: UserWarning: Any labeled images will be returned as a boolean array. Did you mean to use a boolean array?
  return func(*args, **kwargs)
c:\Users\gkristin.ATKK\AppData\Local\miniconda3\envs\cellpose4\Lib\site-packages\skimage\_shared\utils.py:386: UserWarning: Any labeled images will be returned as a boolean array. Did you mean to use a boolean array?
  return func(*args, **kwargs)


In [39]:
mark_boundaries = [m*b for m,b in zip(removed_small_holes,boundaries)]
relabeled = [morphology.label(m,background=0, connectivity=m.ndim) for m in mark_boundaries]

In [ ]:
footprint = morphology.footprint_rectangle((3,3,3))
closing_test = [morphology.closing(m,footprint=footprint,mode='ignore') for m in removed_small_obj] # creates new values

In [34]:
expand_labels = [segmentation.expand_labels(m,distance=1) for m in removed_small_obj]

In [41]:
for i in range(0,3):
    v.add_image(imgs[i],name=f'Img {i}',scale=pixel_spacing)
    v.add_labels(relabeled[i],name=f'New Mask {i}',scale=pixel_spacing)
    v.add_labels(removed_small_obj[i],name=f'OG Mask {i}',scale=pixel_spacing)